# 01. EDA — ESS 배터리 수명 예측

- 데이터: MIT-Stanford Battery Dataset (Severson et al., Nature Energy 2019)
- 목표: Batch 1 / 2 / 3의 특성을 비교하고, 그 결과를 근거로 모델 설계 전략을 세운다.

## 0. 환경 설정

**목적**: 분석에 쓰는 라이브러리, 경로, 그래프 설정, 난수 시드를 한곳에 모아 재현 가능하게 한다.

In [9]:
import os
import logging
import warnings

import numpy as np
import pandas as pd
import mat73
import scipy.io as sio
import matplotlib.pyplot as plt
import matplotlib.cm as cm

# 경고 및 mat73의 string 타입 로딩 메시지 숨김
warnings.filterwarnings('ignore')
logging.getLogger().setLevel(logging.CRITICAL)

# 재현성
SEED = 42
np.random.seed(SEED)

# 경로
DATA_DIR = '../data'
PROC_DIR = '../data/processed'
FIG_DIR = '../results/figures'
os.makedirs(PROC_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

# 그래프 설정 (한글 폰트 포함)
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12

# 표 출력 설정
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

## 1. 데이터 준비

**목적**: 세 배치의 사이클별 요약(`summary`)을 하나의 DataFrame으로 모은다.
원본 파일은 배치당 2~3GB라 매번 읽기 어려우므로, 추출 결과를 파일로 저장해 이후에는 그 파일만 사용한다.

In [10]:
def load_mat(path):
    """.mat 파일 로더. v7.3(HDF5)은 mat73, 그 이하는 scipy로 읽는다."""
    try:
        data = mat73.loadmat(path)
    except Exception:
        data = sio.loadmat(path, simplify_cells=True)
    return data


def to_list_of_dicts(d):
    """mat73의 dict-of-lists 구조를 list-of-dicts로 변환한다."""
    if isinstance(d, dict):
        keys = list(d.keys())
        n = len(d[keys[0]])
        return [{k: d[k][i] for k in keys} for i in range(n)]
    return d


def extract_summary(batch, batch_no):
    """배치 내 모든 셀의 summary를 사이클 단위 DataFrame으로 변환한다."""
    records = []
    for i, cell in enumerate(batch):
        s = cell['summary']
        qd = np.array(s['QDischarge'])
        qc = np.array(s['QCharge'])
        ir = np.array(s['IR'])
        tmax = np.array(s['Tmax'])
        tavg = np.array(s['Tavg'])
        tmin = np.array(s['Tmin'])
        ct = np.array(s['chargetime'])

        cycle_life = float(cell['cycle_life'])
        policy = str(cell.get('policy_readable') or cell.get('policy') or 'unknown')

        for c in range(len(qd)):
            records.append({
                'batch': batch_no,
                'cell_id': f'b{batch_no}c{i}',
                'cycle': c + 1,
                'cycle_life': cycle_life,
                'charging_policy': policy,
                'QD': qd[c],
                'QC': qc[c],
                'IR': ir[c],
                'Tmax': tmax[c],
                'Tavg': tavg[c],
                'Tmin': tmin[c],
                'chargetime': ct[c],
            })
    return pd.DataFrame(records)

In [11]:
FILES = {
    1: '2017-05-12_batchdata_updated_struct_errorcorrect.mat',
    2: '2018-02-20_batchdata_updated_struct_errorcorrect.mat',
    3: '2018-04-12_batchdata_updated_struct_errorcorrect.mat',
}

dfs = []
for b, fname in FILES.items():
    print(f'Batch {b} 로딩 중...')
    mat = load_mat(os.path.join(DATA_DIR, fname))
    batch = to_list_of_dicts(mat['batch'])
    dfs.append(extract_summary(batch, b))
    del mat, batch

df = pd.concat(dfs, ignore_index=True)
df.to_parquet(os.path.join(PROC_DIR, 'summary.parquet'))

print(df.shape)
print(df.groupby('batch')['cell_id'].nunique())

Batch 1 로딩 중...
Batch 2 로딩 중...
Batch 3 로딩 중...
(116722, 12)
batch
1    46
2    47
3    46
Name: cell_id, dtype: int64


**결과 해석**
- 세 배치를 합친 summary는 116,722행(사이클 단위), 12열이다.
- 셀 수는 Batch 1 46개, Batch 2 47개, Batch 3 46개로 총 139개다.
- 추출 결과를 `data/processed/summary.parquet`으로 저장했다. 이후 분석은 이 파일에서 시작한다.

In [12]:
df = pd.read_parquet(os.path.join(PROC_DIR, 'summary.parquet'))
df.head()

,batch,cell_id,cycle,cycle_life,charging_policy,QD,QC,IR,Tmax,Tavg,Tmin,chargetime
0,1,b1c0,1,1190.0,3.6C(80%)-3.6C,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,1,b1c0,2,1190.0,3.6C(80%)-3.6C,1.070689,1.071042,0.016742,35.652016,31.875011,29.566130,13.341250
2,1,b1c0,3,1190.0,3.6C(80%)-3.6C,1.071900,1.071674,0.016724,35.692978,31.931490,29.604385,13.425777
3,1,b1c0,4,1190.0,3.6C(80%)-3.6C,1.072510,1.072304,0.016681,35.680588,31.932603,29.744202,13.425167
4,1,b1c0,5,1190.0,3.6C(80%)-3.6C,1.073174,1.072970,0.016662,35.728691,31.959322,29.644709,13.341442


## 2. Q1 — Cycle Life 분포는 어떻게 생겼는가?

**목적**: 학습 배치(Batch 1)와 테스트 배치(Batch 2, 3)의 수명 분포를 비교한다.
- 분포의 범위와 형태는 배치마다 같은가
- 장수명(>1,000) / 단수명(<500) 셀의 비율은 어떤가
- 유독 짧거나 값이 이상한 셀이 있는가

In [13]:
cells = df.drop_duplicates('cell_id')[['batch', 'cell_id', 'cycle_life', 'charging_policy']].reset_index(drop=True)

print('수명 값이 없는 셀 수')
print(cells[cells['cycle_life'].isna()].groupby('batch').size())

cells.groupby('batch')['cycle_life'].describe().round(1)

수명 값이 없는 셀 수
batch
2    8
3    2
dtype: int64


,count,mean,std,min,25%,50%,75%,max
batch,,,,,,,,
1,46.0,844.7,184.6,534.0,703.2,858.5,914.2,1227.0
2,39.0,565.7,222.2,392.0,439.5,472.0,508.5,1186.0
3,44.0,1059.7,313.9,541.0,828.0,1005.5,1155.2,1935.0
